# Prodigy InfoTech — Data Science Internship
## Task 03: Decision Tree Classifier

**Objective:** Build a decision tree classifier to predict whether a customer will subscribe to a product/service based on demographic and behavioral information.

This project uses the UCI Bank Marketing dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
print('Libraries imported successfully.')

In [ ]:
import urllib.request, zipfile, io
url='https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip'
with urllib.request.urlopen(url) as r: data=io.BytesIO(r.read())
with zipfile.ZipFile(data) as z:
    with z.open('bank-additional/bank-additional-full.csv') as f:
        df=pd.read_csv(f,sep=';')
print('Dataset shape:',df.shape)
display(df.head())

In [ ]:
print(df.columns.tolist())
print('\nTarget distribution:')
display(df['y'].value_counts())
print('\nMissing values:')
display(df.isnull().sum().sort_values(ascending=False).head())

## Prepare the data

`y` is the target. `duration` is excluded because it is only known after a call and would not be available for a pre-call prediction.

In [ ]:
X=df.drop(columns=['y','duration'])
y=df['y'].map({'no':0,'yes':1})
cat=X.select_dtypes(include=['object']).columns.tolist()
num=X.select_dtypes(exclude=['object']).columns.tolist()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
pre=ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),cat),('num','passthrough',num)])
model=DecisionTreeClassifier(max_depth=5,random_state=42,class_weight='balanced')
pipeline=Pipeline([('preprocessor',pre),('classifier',model)])
pipeline.fit(X_train,y_train)
print('Model trained successfully.')

In [ ]:
y_pred=pipeline.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test,y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test,y_pred,target_names=['No Purchase','Purchase']))

In [ ]:
cm=confusion_matrix(y_test,y_pred)
disp=ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=['No Purchase','Purchase'])
fig,ax=plt.subplots(figsize=(7,6)); disp.plot(ax=ax); plt.title('Decision Tree Confusion Matrix'); plt.tight_layout(); plt.show()

In [ ]:
features=pipeline.named_steps['preprocessor'].get_feature_names_out()
tree=pipeline.named_steps['classifier']
plt.figure(figsize=(22,10))
plot_tree(tree,feature_names=features,class_names=['No Purchase','Purchase'],filled=True,max_depth=3,fontsize=8)
plt.title('Decision Tree Classifier (First 3 Levels)'); plt.show()

## Conclusion

The Decision Tree classifier predicts whether a customer subscribes to a term deposit. Categorical variables are one-hot encoded, the data is split into training and test sets, and the model is evaluated with accuracy, a classification report, and a confusion matrix. Because the target classes are imbalanced, accuracy should be considered together with the other metrics.